# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [2]:

import pandas as pd
import plotly.express as px
import numpy as np
from tkinter import Tk, filedialog

# Open a window to select the CSV file
root = Tk()
root.withdraw()
root.attributes("-topmost", True)

file_path = filedialog.askopenfilename(
    title="Select global_energy_mix.csv",
    filetypes=[("CSV Files", "*.csv")]
)

root.destroy()

# Load dataset
if not file_path:
    raise FileNotFoundError("No file was selected.")

df = pd.read_csv(file_path)

# Source type mapping
source_category = {
    'Coal': 'Fossil',
    'Oil': 'Fossil',
    'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon',
    'Hydro': 'Low-carbon',
    'Wind': 'Renewable',
    'Solar': 'Renewable',
    'Other Renewables': 'Renewable'
}

df['Source_Type'] = df['Source'].map(source_category)

print("Dataset loaded successfully!")
print(f"Loaded: {len(df)} rows")
display(df.head(10))


Dataset loaded successfully!
Loaded: 144 rows


,Region,Country,Source,TWh,Source_Type
0,Asia,China,Coal,1116.19,Fossil
1,Asia,China,Oil,802.31,Fossil
2,Asia,China,Natural Gas,851.77,Fossil
3,Asia,China,Nuclear,113.22,Low-carbon
4,Asia,China,Hydro,219.88,Low-carbon
5,Asia,China,Wind,100.75,Renewable
6,Asia,China,Solar,84.13,Renewable
7,Asia,China,Other Renewables,36.13,Renewable
8,Asia,India,Coal,496.61,Fossil
9,Asia,India,Oil,495.98,Fossil


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [3]:


df_fossil = df.loc[
    df["Source_Type"] == "Fossil"
].copy()

# Aggregate by region
region_totals = (
    df_fossil.groupby("Region")["TWh"]
    .sum()
    .sort_values(ascending=False)
)

top_region = region_totals.index[0]

# Colourblind-friendly colours
fossil_colors = {
    "Coal": "#0072B2",
    "Oil": "#E69F00",
    "Natural Gas": "#009E73"
}

fig1 = px.treemap(
    df_fossil,
    path=["Region", "Country", "Source"],
    values="TWh",
    color="Source",
    color_discrete_map=fossil_colors,
    title=f"{top_region} has the highest total fossil energy"
)

# Make parent nodes grey
for trace in fig1.data:
    colors = list(trace.marker.colors)

    for i, node_id in enumerate(trace.ids):
        if len(node_id.split("/")) < 3:
            colors[i] = "#BDBDBD"

    trace.marker.colors = colors

fig1.update_traces(
    texttemplate="%{label}<br>%{value:,.1f} TWh",
    hovertemplate="%{label}: %{value:,.1f} TWh<extra></extra>"
)

fig1.update_layout(
    height=650,
    margin=dict(t=80, l=20, r=20, b=20)
)

fig1.show()

print("Fossil energy by region:")
display(region_totals)


Fossil energy by region:


Region
Asia             4781.40
North America    3337.53
Europe            791.46
South America     593.91
Africa            520.02
Oceania           334.74
Name: TWh, dtype: float64

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [4]:


df_low = df.loc[
    df["Source_Type"] == "Low-carbon"
].copy()

# Aggregate TWh by country
low_country = (
    df_low.groupby("Country", as_index=False)["TWh"]
    .sum()
    .sort_values("TWh", ascending=False)
)

low_country["All"] = "Low-carbon"

top_country = low_country.iloc[0]["Country"]
top_twh = low_country.iloc[0]["TWh"]

# CHART 1: TREEMAP
fig2 = px.treemap(
    low_country,
    path=["All", "Country"],
    values="TWh",
    color_discrete_sequence=["#0072B2"],
    title="Low-carbon Energy by Country (TWh)"
)

fig2.update_traces(
    texttemplate="%{label}<br>%{value:,.1f} TWh"
)

fig2.update_layout(height=600)
fig2.show()

# CHART 2: HORIZONTAL BAR CHART
fig3 = px.bar(
    low_country,
    x="TWh",
    y="Country",
    orientation="h",
    text="TWh",
    color_discrete_sequence=["#0072B2"],
    title=f"{top_country} leads low-carbon energy with {top_twh:,.1f} TWh"
)

fig3.update_traces(
    texttemplate="%{x:,.1f} TWh",
    textposition="outside"
)

fig3.update_layout(
    xaxis_title="Low-carbon Energy (TWh)",
    yaxis_title="Country",
    yaxis=dict(categoryorder="total ascending"),
    height=max(500, len(low_country) * 30),
    margin=dict(r=100)
)

fig3.show()

display(low_country)


,Country,TWh,All
17,United States,463.57,Low-carbon
5,China,333.10,Low-carbon
10,India,170.15,Low-carbon
9,Germany,166.98,Low-carbon
2,Brazil,151.47,Low-carbon
8,France,125.20,Low-carbon
3,Canada,122.40,Low-carbon
16,United Kingdom,118.40,Low-carbon
11,Japan,84.17,Low-carbon
13,Mexico,73.70,Low-carbon


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [5]:


region_source = (
    df_fossil.groupby(
        ["Region", "Source"],
        as_index=False
    )["TWh"].sum()
)

# Calculate total fossil energy per region
region_source["Region_Total"] = (
    region_source.groupby("Region")["TWh"]
    .transform("sum")
)

# Calculate percentage share
region_source["Share"] = (
    region_source["TWh"] /
    region_source["Region_Total"] * 100
)

# Find coal share by region
coal_shares = (
    region_source[
        region_source["Source"] == "Coal"
    ]
    .set_index("Region")["Share"]
    .sort_values(ascending=False)
)

top_coal_region = coal_shares.index[0]
top_coal_share = coal_shares.iloc[0]

# Coal coloured, other fuels grey
stack_colors = {
    "Coal": "#0072B2",
    "Oil": "#BDBDBD",
    "Natural Gas": "#737373"
}

fig4 = px.bar(
    region_source,
    x="Share",
    y="Region",
    color="Source",
    orientation="h",
    barmode="stack",
    category_orders={
        "Source": ["Coal", "Oil", "Natural Gas"],
        "Region": coal_shares.index[::-1].tolist()
    },
    color_discrete_map=stack_colors,
    title=(
        f"{top_coal_region} has the highest coal share "
        f"({top_coal_share:.1f}%)"
    ),
    labels={
        "Share": "Share of Fossil Energy (%)",
        "Region": "Region"
    }
)

# Add percentage labels to Coal only
for trace in fig4.data:
    if trace.name == "Coal":
        trace.text = [
            f"{value:.1f}%" for value in trace.x
        ]
        trace.texttemplate = "%{text}"
        trace.textposition = "inside"
    else:
        trace.text = None
        trace.texttemplate = None

fig4.update_layout(
    height=550,
    xaxis=dict(range=[0, 100]),
    legend_title="Energy Source",
    bargap=0.25
)

fig4.show()

print("Coal dependency by region:")
display(coal_shares.round(2))


Coal dependency by region:


Region
Africa           41.10
Asia             39.40
Oceania          38.06
North America    23.52
Europe           22.31
South America    10.73
Name: Share, dtype: float64

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The Task 1 treemap answers which countries and fossil fuel sources account for the largest absolute energy consumption in TWh. It emphasizes the hierarchy and relative sizes of the energy totals.

The Task 3 stacked bar chart answers which region has the highest percentage of coal in its fossil energy mix. Because each bar represents 100% of a region's fossil energy, it makes comparisons of coal dependency between regions easier.